# The Order Book and Order Types

When your code sends an order, it lands in a precise data structure called the order book, and the type of order you send decides how — and whether — it fills. Getting this wrong is one of the most expensive beginner errors: a careless market order in a thin book can cost more than a week of edge. This lesson makes the order book concrete and walks through every order type you'll actually use.

By the end of this lesson you will be able to:

- Read a limit order book, including bids, asks, and depth
- Distinguish market, limit, stop, and stop-limit orders and when to use each
- Explain IOC and FOK time-in-force conditions
- Define maker versus taker and why the distinction affects your costs
- Trace how a large order "walks the book" and what that does to your fill price



## 1. The Limit Order Book

The **limit order book (LOB**)** is simply the exchange's sorted list of all resting limit orders for an instrument. It has two sides:

1. **Bids** - Orders to buy , sorted from highest price(best) down.
2. **Asks** - Orders to sell , sorted from lowest price(best) up.

Each level shows a price and the total **size**(quantity) resting there. The total quantity available across levels is the books depth. Here is a small book:

```
        Price    Size
Asks    50.09    400
        50.08    250
        50.07    150     <- best ask (lowest sell)
------------------------ spread = 0.03
Bids    50.04    300     <- best bid (highest buy)
        50.03    500
        50.02    220
```

The **best bid** (50.04) and **best ask** (50.07) define the top of book. The gap between them — 0.03 here — is the **spread**. The midpoint, 50.055, is often used as a "fair" reference price, but note you can't actually trade at the mid with a simple order. Depth matters: this book can absorb 150 shares of buying at 50.07 before the price you pay rises.

The best bid (50.04) and best ask (50.07) define the top of book. The gap between them — 0.03 here — is the spread. The midpoint, 50.055, is often used as a "fair" reference price, but note you can't actually trade at the mid with a simple order. Depth matters: this book can absorb 150 shares of buying at 50.07 before the price you pay rises.

## 2. Order Types

### 1. Market Order:
A market order says "fill me immediately at the best available price, whatever it is." It guarantees execution but not price. It always crosses the spread — buying lifts the ask, selling hits the bid. In a deep book this is fine; in a thin one it can fill far from where you expected. Never assume you'll get the price on your screen.

### 2. Limit Order

A limit order says "fill me only at this price or better." A buy limit at 50.05 will not pay more than 50.05. It guarantees price but not execution — if the market never reaches your limit, you simply don't trade. A limit order priced away from the market rests in the book and adds liquidity; a limit order priced to cross executes immediately like a market order but with a price cap.

### 3. Stop Order

A stop order is dormant until the market touches a trigger price, then becomes a market order. A common use is a protective stop-loss: "if price falls to 49.50, sell at market." Stops are not visible in the book until triggered. The danger: once triggered they become market orders, so in a fast drop your fill can be well below the stop level.


### 4. Stop Limit Order

A stop-limit combines the two: when the trigger is hit, it submits a limit order rather than a market order. "If price falls to 49.50, place a sell limit at 49.40." This caps how bad your fill can be — but if price gaps straight through 49.40, the limit may never fill and you're left holding the position. You trade execution certainty for price protection.


>Rule of thumb: market and stop orders prioritise getting done; limit and stop-limit orders prioritise price. You can rarely have both at once.

A compact way to hold the four order types in your head is a 2×2 grid along two questions: does it trigger on a condition? and does it cap the price?


```
                    no price cap        price cap
not triggered:      market order        limit order
triggered:          stop order          stop-limit order

```

A market order is the "just do it" choice; a limit order adds a price ceiling/floor; a stop is a market order that waits for a trigger; a stop-limit is a limit order that waits for a trigger. Every order you place is one of these four combinations, and choosing correctly is choosing which of execution-certainty or price-certainty you're willing to give up.

## 3. Time-in-force: IOC and FOK
Beyond price, you can control how long an order lives:

- **IOC (Immediate-Or-Cancel)**. Fill whatever you can right now, cancel the rest. Useful for taking available liquidity without leaving a resting order behind.
- **FOK (Fill-Or-Kill)**. Fill the entire quantity immediately or cancel the whole thing. No partial fills. Useful when a partial position is worse than none.
- **Day / GTC**. A plain limit order usually rests until the end of the day (Day) or until you cancel it (Good-Til-Canceled).

The distinction between IOC and FOK matters more than it looks. Suppose you want 1,000 shares but only 600 are available at your price. An IOC takes the 600 and cancels the rest — you end up with a partial position. A FOK sees it can't get all 1,000 and cancels entirely — you end up with nothing. Which you want depends on the strategy: if a partial fill leaves you with awkward, unhedged risk (say, one leg of a pair trade), FOK protects you; if any fill is better than none, IOC is right. Beginners often leave time-in-force at the default and are then surprised by partial fills; choosing deliberately is part of execution discipline.

## 4. Maker Vs Taker

This distinction drives your fees and your strategy design:
- A **Maker** posts a resting limit order that adds liquidity to the book. Someone else later trades against it.
- A **Taker**  sends an order that removes liquidity by matching against a resting order (any market order, or a marketable limit).

Most exchanges charge takers a fee and often pay makers a small rebate, because makers provide the liquidity everyone relies on. For a high-frequency strategy, the difference between paying a taker fee and earning a maker rebate can be the entire difference between profit and loss. The catch: maker orders aren't guaranteed to fill, so you trade fee savings for execution uncertainty.

Make the economics concrete. Suppose an exchange charges takers 5 basis points and pays makers a 2 bp rebate. On a $50,000 trade, taking costs you 0.0005 50000 = $25, while making earns you 0.0002 50000 = $10 — a $35 swing on a single trade purely from which side of the book you're on. For a strategy trading hundreds of times a day, that swing is the whole business. This is why serious short-horizon strategies are designed around posting liquidity and patiently waiting in the queue rather than crossing the spread. The price of that rebate is real, though: your resting order might never fill, or fill only after the market has moved against the reason you wanted in.



## 5. How orders fill against the book

A resting limit order fills under price-time priority: better prices fill first, and at the same price, the order that arrived earlier fills first. So if you post a buy limit at 50.04 behind 300 shares already resting at 50.04, those 300 shares must trade before you do.

A marketable order consumes the book from the best price outward, level by level, until it is filled or runs out of book.

This "queue position" has real consequences for maker strategies. When you post at the best bid behind 300 shares, you only get filled after all 300 ahead of you trade and a seller is still willing to hit that price. If the price ticks up before your turn arrives, your order never fills and you've missed the move; if it ticks down, you get filled right as the market turns against you — a phenomenon traders call "adverse selection." Posting passively is not free money; you're compensated with a rebate precisely because you bear the risk of filling only when it's unfavorable.

## 6. Working Example: Walking the Book

Take the book from earlier and send a **market buy for 600 shares**. The engine consumes the asks from cheapest up:


```

Fill 150 @ 50.07
Fill 250 @ 50.08
Fill 200 @ 50.09   (200 of the 400 resting at this level)

```

You need 600 Shares

1. 150 shares at 50.07 (best ask exhausted)
2. 250 shares at 50.08 (next level exhausted)
3. 200 shares at 50.09 (200 of the 400 here)

`Average price = (15050.07 + 25050.08 + 200*50.09) / 600`. Here is the calculation in Python:



In [1]:
fills = [(150, 50.07), (250, 50.08), (200, 50.09)]
total_shares = sum(q for q, p in fills)
total_cost = sum(q * p for q, p in fills)
avg_price = total_cost / total_shares


print(total_shares)
print(round(avg_price,4))


600
50.0808


You wanted the 50.07 best ask, but you actually paid an average of 50.0792. The extra 50.0792 - 50.07 = 0.0092 per share is **slippage from walking the book** — the direct cost of demanding more liquidity than sits at the top level. Your order also pushed the best ask up to 50.09, moving the market against the next buyer. A patient limit order resting at 50.07 might have avoided this entirely — at the risk of not filling.

## Worked Example: Comparing a market order to marktable limit
Worked example: comparing a market order to a marketable limit
A common, smarter alternative to a raw market order is a marketable limit — a limit order priced aggressively enough to execute now, but with a cap that protects you from a catastrophic fill if the book is thinner than you thought. Let's compare on the same book, sending 600 shares but capping the price at 50.08.

In [3]:
asks = [(50.07, 150), (50.08, 250), (50.09, 400)]  # (price, size) cheapest first

def markatable_limit_buy(ask, qty, limit_price):
    remaining, cost, filled = qty, 0.0,0
    for price, size in asks:
        if price > limit_price:
            break
        take  = min(remaining, size)
        cost += take * price
        filled += take
        remaining -= take
        if remaining == 0:
            break
    avg = cost/filled if filled else None
    return filled, avg, remaining

filled, avg, unfilled = markatable_limit_buy(asks, 600, 50.08)
print(f"filled {filled} @ {avg:.4f}, unfilled {unfilled}")
# filled 400 @ 50.0763, unfilled 200

filled 400 @ 50.0763, unfilled 200


With the 50.08 cap, you fill 400 shares at a better average (50.0763) and the remaining 200 simply don't execute, because filling them would have meant paying 50.09 — above your cap. You've *traded completeness* for price protection: a pure market order would have filled all 600 at 50.0792, but a sudden gap in the book (say the 50.09 level were actually at 51.00) could have cost you dearly. The marketable limit is the workhorse of careful execution: it gets you done now when liquidity is there, while putting a hard floor under how badly you can be filled.

## Common Mistakes

- **Using market orders in thin books**. With little depth, a market order walks several levels and fills far from the screen price.
- **Confusing stop with stop-limit**. A stop becomes a market order (fills, maybe badly); a stop-limit may not fill at all in a fast move.
- **Forgetting maker/taker fees in the backtest**. A strategy profitable assuming maker rebates can be a loser if it actually has to cross the spread as a taker.
- **Assuming limit orders always fill**. A resting limit only fills if the market reaches it and the orders ahead of you clear first.
- **Ignoring depth**. Looking only at the best bid/ask hides the fact that your size may need three levels to fill.